In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_movies")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlproyecto12")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/FilmDetails.csv"

In [0]:
df_details = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
details_schema = StructType(fields=[
    StructField("id", IntegerType(), False),
    StructField("director", StringType(), True),
    StructField("top_billed", StringType(), True),
    StructField("budget_usd", StringType(), True),
    StructField("revenue_usd", StringType(), True)
])

In [0]:
df_detail_final = spark.read\
.option('header', True)\
.option('escape', '"')\
.schema(details_schema)\
.csv(ruta)

In [0]:
display(df_detail_final.limit(5))

In [0]:
details_final_df = df_detail_final.select(
            col("id").alias("movie_id"),
            col("director"),
            col("top_billed"),
            col("budget_usd"),
            col("revenue_usd")
        ).withColumn("ingestion_date", current_timestamp())

In [0]:
details_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.moviedetails")

In [0]:
%sql
select count(1) from catalog_movies.bronze.moviedetails